# Script de treinamento

### Carregamento das bibliotecas requeridas

In [1]:
import os
'''
estou usando WSL do windows e preciso compilar o kernel linux de novo se não ficam aparecendo vários warnings,
essas próximas 3 linhas de comando são para silencia-los
'''
import logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0 (default) = all messages, 1 = INFO messages, 2 = INFO and WARNING messages, 3 = only ERROR messages
logging.getLogger('tensorflow').setLevel(logging.ERROR)  # Suppress TensorFlow INFO and WARNING messages

import tensorflow as tf
import keras
import importlib
import argparse

# essas libs foram as criadas por mim
from utils.utils import *
from utils.data_lists import *
from utils.callbacks import get_callbacks
from dataloader.dataloader import *
from models.data_augmentation import *
from models.deep_lstm import build_model

2024-03-03 11:36:21.768592: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-03 11:36:21.768632: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-03 11:36:21.770327: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


### Parâmetros básicos, como nome do modelo, caminho das configurações e qual sujeito treinar

In [2]:
subject_number = 5
model_name = 'transformers'
config_path = 'configs/config.yml'

### Carregamento das configurações e do modelo

In [3]:
# Importing model module
'''
dessa forma achei mais fácil, porque mudando o nome do modelo, você usa esse
mesmo script para fazer o treinamento
'''
try:
    model_module = importlib.import_module(f"models.{model_name}")
except ImportError as exc:
    raise ValueError(f"Model '{model_name}' not found: {exc}") from exc
# Loads model configuration parameters from a YAML file.
try:
    config = load_config(config_path)
except ValueError as exc:
    print(f"Error: {exc}")

model_params = config["model_params"][model_name]
model_params_dict = {}
for param_dict in model_params:
    model_params_dict.update(param_dict)

### Carregamento dos dados para treinamento

In [4]:
subjects = []
base_path = config["dataset_path"]

# Create the path for the dataset, based on subject number
subject_path1 = f"subject{subject_number:02d}_session1/"
subject_path2 = f"subject{subject_number:02d}_session2/"

# Path of the first session
session1_path = os.path.join(base_path, subject_path1)

# Path of the second session
session2_path = os.path.join(base_path, subject_path2)

subjects.append([session1_path, session2_path])

### Criação do modelo

In [5]:
# Build the model base architecture
backbone_model = model_module.build_model(**model_params_dict)
estimator_model = build_model(embed_dim=256)

### Carregamento dos parâmetros e resumo do modelo

In [6]:
# Model Parameters
BATCH_SIZE = config["batch_size"]
INPUT_SHAPE_0 = config["input_shape"][0]
INPUT_SHAPE_1 = config["input_shape"][1]
OUTPUT_SHAPE = config["output_shape"]
EPOCHS = config["epoch"]

# Build the final model
inputs = tf.keras.layers.Input(shape=(100, 256))
# Prepare the data for fft
reshape = tf.keras.layers.Reshape((100, 256, 1))
inputs2 = reshape(inputs)
fft_inputs = tf.cast(
    inputs, dtype=tf.complex64, name=None
)
# Separate the complex and the real part in two different channels
complex_inputs = fft_tensor(fft_inputs)
latent = backbone_model(complex_inputs)
outputs = estimator_model(latent)
model = tf.keras.Model(inputs, outputs)

# View information about the model
print(model.summary())

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 100, 256)]           0         []                            
                                                                                                  
 tf.cast (TFOpLambda)        (None, 100, 256)             0         ['input_3[0][0]']             
                                                                                                  
 tf.signal.fft2d (TFOpLambd  (None, 100, 256)             0         ['tf.cast[0][0]']             
 a)                                                                                               
                                                                                                  
 tf.math.real (TFOpLambda)   (None, 100, 256)             0         ['tf.signal.fft2d[0][0]'

###  Data augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.GaussianNoise(stddev=0.02),
    AmplitudeScalingLayer(scale_range=(0.8, 1.2)),
    TimeShiftLayer(max_shift=0.2),
    PhaseRotationLayer(rotation_range=(-pi/9, pi/9)),
    layers.Dropout(0.2),
])

model = tf.keras.Sequential([
    data_augmentation,
    model,
])

In [7]:
# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
'''
sse é a soma de todos os erros, eu usei por que achei mais fácil acompanhar o treinamento e saber que está evoluindo,
mas estatisticamente ela não diz nada, já que a soma do erro pode cair, mas a predição do sinal ainda ser ruim.
a loss foi criação minha, é uma loss ponderada pela potência do sinal e o MSE, eu multiplico o MSE pela potência,
dessa forma ele pesa mais o erro do canal que o sinal é ativado.
'''
model.compile(optimizer=optimizer, loss=loss, metrics=['mse', 'mae', sse])

## Geração dos datasets

In [8]:
# Base folder for the model logs
base_dir = f"logs/models/{model_name}/wa_all_fingers/"

# Check if the base directory exists and is a directory
if not os.path.exists(base_dir):
    os.makedirs(base_dir)
if os.path.exists(base_dir) and os.path.isdir(base_dir):
    # List all directories inside the base directory
    sub_dirs = os.listdir(base_dir)

    # Loop over the found directories
    for sub_dir in sub_dirs:
        # Check if the directory is a valid directory (not a file)
        if os.path.isdir(os.path.join(base_dir, sub_dir)):
            try:
                # Try to convert the directory name to a subject number
                found_subject_number = int(sub_dir.split('_')[1])
                if found_subject_number == subject_number:
                    print(f"Subject {subject_number} already trained. Skipping.")
            except ValueError:
                # If unable to convert to number, skip this directory
                pass

Subject 5 already trained. Skipping.


In [9]:
# Create a path for the dataset
path1 = [subjects[0][0]]
path2 = [subjects[0][1]]
path3 = [subjects[0][1]]

In [10]:
'''
eu fiz uma lista de como os dados são arrumados, fica na pasta util/data_lists.py
aí faço iterações nesse arquivo, assim o código fica mais legível
'''
# Generate the Training Dataset
train_dataset = (
    tf.data.Dataset.from_generator(
        lambda: create_synced_datasets_generator(
            path1, emg_tr_files_lists, force_tr_files_lists
        ),
        output_signature=(
            tf.TensorSpec(
                shape=(INPUT_SHAPE_0, INPUT_SHAPE_1), dtype=tf.float32
            ),
            tf.TensorSpec(shape=(OUTPUT_SHAPE,), dtype=tf.float32),
        ),
    )
    .shuffle(buffer_size=BATCH_SIZE)
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
)

# Generate the Validation Dataset
val_dataset = (
    tf.data.Dataset.from_generator(
        lambda: create_synced_datasets_generator(
            path2, emg_vl_files_lists, force_vl_files_lists
        ),
        output_signature=(
            tf.TensorSpec(
                shape=(INPUT_SHAPE_0, INPUT_SHAPE_1), dtype=tf.float32
            ),
            tf.TensorSpec(shape=(OUTPUT_SHAPE,), dtype=tf.float32),
        ),
    )
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
)

# Generate the Test Dataset
test_dataset = (
    tf.data.Dataset.from_generator(
        lambda: create_synced_datasets_generator(
            path3, emg_te_files_lists, force_te_files_lists
        ),
        output_signature=(
            tf.TensorSpec(
                shape=(INPUT_SHAPE_0, INPUT_SHAPE_1), dtype=tf.float32
            ),
            tf.TensorSpec(shape=(OUTPUT_SHAPE,), dtype=tf.float32),
        ),
    )
    .batch(32)
    .prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
)

## Treinamento do modelo

In [ ]:
# Load callbacks
'''
callbacks são funções que automatizam algumas partes 
do treinamento
coloquei elas em um script, dentro de utils/callbacks
para deixar o código mais legível
'''
callbacks = get_callbacks(model_name, subject_number)

# Fit data to model
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=callbacks,
)

Epoch 1/200


I0000 00:00:1709476593.470679   63567 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


    281/Unknown - 133s 439ms/step - loss: 0.0283 - mse: 0.0095 - mae: 0.0543 - sse: 6.0513
Epoch 1: val_loss improved from inf to 0.03167, saving model to logs/models/transformers/na_all_fingers/subject_5/
281/281 [==============================] - 166s 558ms/step - loss: 0.0283 - mse: 0.0095 - mae: 0.0543 - sse: 6.0513 - val_loss: 0.0317 - val_mse: 0.0056 - val_mae: 0.0476 - val_sse: 3.5554 - lr: 0.0010
Epoch 2/200
281/281 [==============================] - ETA: 0s - loss: 0.0381 - mse: 0.0104 - mae: 0.0623 - sse: 6.6206
Epoch 2: val_loss did not improve from 0.03167
281/281 [==============================] - 58s 205ms/step - loss: 0.0381 - mse: 0.0104 - mae: 0.0623 - sse: 6.6206 - val_loss: 0.0335 - val_mse: 0.0061 - val_mae: 0.0439 - val_sse: 3.8923 - lr: 0.0010
Epoch 3/200
281/281 [==============================] - ETA: 0s - loss: 0.0445 - mse: 0.0134 - mae: 0.0697 - sse: 8.5636
Epoch 3: val_loss did not improve from 0.03167
281/281 [==============================] - 32s 112ms/step

In [ ]:
# Plot training & validation loss values
fig, ax = plt.subplots(figsize=(16, 8))
ax.set_title('Loss')
ax.plot(history.history['loss'], label = 'train')
ax.plot(history.history['val_loss'], label = 'test')
ax.set_ylabel('Loss')
ax.set_xlabel('Epoch')
ax.legend(['Train', 'Validation'])

## Validação do modelo no dataset de teste

In [ ]:
# Calculate the force
predictions, force_signal, emg_signal = [], [], []
for emg, f in test_dataset:
    prediction = model.predict(emg, verbose=0)
    predictions.append(prediction)
    force_signal.append(f)
    emg_signal.append(emg)
# Converter a lista de previsões em um tensor
predictions = tf.concat(predictions, axis=0)
force_signal = tf.concat(force_signal, axis=0)

In [ ]:
# Define the length of the signal
signal_length = len(force_signal)

# Define the segments
all_lengths = []
x = 299
y = 2395
for i in range(5):
    segment_lengths = [x, 299, 598, 598, 299, signal_length - y]
    x += 2395
    y += 2395
    all_lengths.append(segment_lengths)

# Define the values for each segment
segment_values = []
for i in range(5):
    # segment_values_dim = [0, np.max(normalized_signal[:, :, i]), np.min(normalized_signal[:, :, i]), np.max(normalized_signal[:, :, i]), 0, 0]
    segment_values_dim = [0, 1, -1, 1, 0, 0]
    segment_values.append(segment_values_dim)

# Generate the signal by linearly interpolating between segment values
target_force_signals = []
# Generate signals and store them in the list
for i in range(5):
    # Generate the signal according to the specified pattern
    signal = np.zeros(signal_length)
    start = 0
    for length, value in zip(all_lengths[i], segment_values[i]):
        end = start + length
        signal[start:end] = np.linspace(signal[start - 1], value, length)
        start = end
    # Append the generated signal to the list
    target_force_signals.append(signal)

## Análise qualitativa

In [ ]:
predicted = np.array(predictions)
real = np.array(force_signal)

# Define a função para converter de samples para segundos
def samples_to_seconds(samples, sample_rate):
    return samples / sample_rate

# Define a taxa de amostragem (sample rate)
sample_rate = 100  # Exemplo, ajuste conforme sua taxa de amostragem

# Create the figure
fig, axes = plt.subplots(5, 1, figsize=(10, 16))
fig.suptitle("", fontsize=18, y=1.0)

# Create subplots based on the GridSpec layout
for i, ax in enumerate(axes):
    # Define a posição vertical do subplot
    sub_plot_position = 3 - i * 1.5  # Decremento de 1.5 para cada subplot

    # Plot base target for force
    ax.plot(target_force_signals[i], color="orangered", label="Target force")

    # Plot real force
    ax.plot(real[:, i], color="blue", label="Real force")

    # Plot predicted force
    ax.plot(predicted[:, i], color="red", label="Predicted force")

    # Configure the subplot
    ax.set_title(f"Finger {i+1}", fontsize=16)  # Adjust title font size
    ax.xaxis.grid(True, which='minor')
    # ax.set_ylabel("Values", fontsize=14)  # Adjust y-label font size
    
    # Hide x-axis labels for all but the bottom subplot
    if i < len(axes) - 1:
        ax.xaxis.set_ticklabels([])

# Set x-label for the bottom subplot
axes[-1].set_xlabel("Seconds", fontsize=14)  # Adjust x-label font size
# Add a text annotation in the middle left of the figure
fig.text(-0.015, 0.5, 'Force Values: % OF MVC', va='center', rotation='vertical', fontsize=14)

# Add legend
axes[0].fill_between([], [], [], color='orange', alpha=0.2, label="Real force dispersion")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right')

# Ajuste o layout para evitar sobreposição
plt.tight_layout()

# Apply time scale for the last plot
last_ax = axes[-1]
last_ax.set_xticklabels(samples_to_seconds(last_ax.get_xticks(), sample_rate))

# Show the plot
plt.show()

## Análise quantitativa

In [ ]:
'''
aqui eu faço um slice só na parte da ativação para o cálculo do mse,
faço isso porque o repouso faz o MSE abaixar muito
'''
def mse_per_channel_slice(y_true, y_pred, slice_size):
    """
    Mean squared error (MSE) per channel for a specified slice.

    Args:
        y_true: True labels (shape: [batch_size, num_channels]).
        y_pred: Predicted labels (shape: [batch_size, num_channels]).
        slice_size: Size of the slice for each channel.

    Returns:
        MSE per channel for the specified slice (shape: [num_channels]).
    """
    # Slice the true and predicted labels for each channel
    y_true_slices = [y_true[:, i][slice_size*i:slice_size*(i+1)] for i in range(y_true.shape[1])]
    y_pred_slices = [y_pred[:, i][slice_size*i:slice_size*(i+1)] for i in range(y_pred.shape[1])]

    # Calculate MSE for each channel slice
    mse_per_channel = np.array([np.mean((true - pred)**2) for true, pred in zip(y_true_slices, y_pred_slices)])
    return mse_per_channel

In [ ]:
# Tamanho do slice para cada canal
slice_size = int(len(force_signal)/5)

# Calcula o MSE para o slice específico
mse_slice = mse_per_channel_slice(real, predicted, slice_size)

print("MSE por canal no slice específico:", mse_slice)

In [ ]:
# Create a violin plot
plt.figure(figsize=(10, 6))
parts = plt.violinplot(mse_slice, widths=0.9, showmeans=True)

# Set the locations of the violins manually
# plt.xticks(np.arange(1, len(subjects) + 1), subjects_number)

# Add labels and title
plt.ylabel('MSE Values: % OF MVC', fontsize=14)
# plt.title('Violin Plot of Data', fontsize=16)

# Show plot
plt.show()